In [1]:
# ── 1. INSTALL ────────────────────────────────────────────────
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl accelerate bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 84.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from kaggle_secrets import UserSecretsClient
import wandb

# 1. Get key
WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")

# 2. Login FIRST
wandb.login(key=WANDB_API_KEY)

# 3. Init AFTER
wandb.init(
    project="content-gen-phi",
    name="phi-lora-768-bf16_posts",
    config={
        "model": "phi-3-mini-4k-instruct",
        "chunk_size": 768,
        "lora_r": 16,
        "batch_size": 4,
        "grad_accum": 1,
        "learning_rate": 2e-4,
        "max_steps": 1000,
        "bf16": True,
    }
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ahmed-farag08023 (ahmed-farag08023-ain-shams-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
# ── 3. LOAD & PREPARE DATA ────────────────────────────────────
import pandas as pd
from sklearn.model_selection import train_test_split

df_posts= pd.read_csv("/kaggle/input/datasets/ahmedmohamedfarag111/posts-new/posts.csv")

In [12]:
df_posts.head()

,Unnamed: 0,post_content,post_title,text
0,0,"I know this post sounds super petty, but this ...",telling my boyfriend I'll shave my legs if he ...,You are a professional social media writer.\n\...
1,1,My daughter Bryn F9 is going on a trip to a ne...,pulling my daughter from a waterpark trip beca...,You are a professional social media writer.\n\...
2,2,Alright so my son (17) has weekly therapy appo...,not letting an elderly woman have my son’s sea...,You are a professional social media writer.\n\...
3,3,We live three blocks away from my parents and ...,taking my kids to my parents house to sleep be...,You are a professional social media writer.\n\...
4,4,My daughter (16) and I have gotten into a mass...,calling my daughter a selfish insecure little ...,You are a professional social media writer.\n\...


In [13]:
from sklearn.model_selection import train_test_split

# Configuration for formatting (can reuse or redefine for clarity)
EOS_TOKEN = '<|end_of_text|>'
MAX_CHARS = 4000 # Keeping consistent with the last MAX_CHARS for posts

# Define the expert prompt template for blogs, removing the Tone requirement
posts_prompt = """You are a professional social media writer.

Title: {0}


Write a social media post that:

* Is engaging and attention-grabbing.
* Sounds human and authentic.
* Fits the Title and context.
* May include emojis, hashtags, or mentions when appropriate.
* Is similar in style to real posts found on social media.

Post:
{1}"""


# Apply the formatting to match  structure, removing the 'Tone' argument
df_posts['text'] = df_posts.apply(
    lambda row: posts_prompt.format(
        row['post_title'],
        row['post_content']
    ) + EOS_TOKEN,
    axis=1
).str.slice(0, MAX_CHARS)

# Optional sample for faster testing
sample_size = 3000 # Keeping consistent with the last sample_size

_sample = (
    df_posts.sample(min(len(df_posts), sample_size), random_state=42)
    .reset_index(drop=True)
)

# Split into Train and Val sets
train_df, val_df = train_test_split(
    _sample,
    test_size=0.15,
    random_state=43
)

print(f"Total processed posts available: {len(df_posts)}")
print(f"Train blog set size: {len(train_df)} | Validation blog set size: {len(val_df)}")

print("\nPreview of formatted text for blog training:")
print(train_df['text'].iloc[0][:500] + "...")

Total processed posts available: 2922
Train blog set size: 2483 | Validation blog set size: 439

Preview of formatted text for blog training:
You are a professional social media writer.

Title: telling my husband his disco party in our house is a no-go?


Write a social media post that:

* Is engaging and attention-grabbing.
* Sounds human and authentic.
* Fits the Title and context.
* May include emojis, hashtags, or mentions when appropriate.
* Is similar in style to real posts found on social media.

Post:
My husband told me that he wants to run a disco-themed party from 10am Friday morning until 4pm Saturday afternoon in our house...


------------------------------------------------------------------

-----------------------------------------------------------------

In [14]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

# ── Model (unchanged) ─────────────────────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length=1024,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=43,
)

==((====))==  Unsloth 2026.6.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


In [15]:
# ── Dataset — raw text only, let Unsloth handle the rest ──────
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["text"]])
val_dataset   = Dataset.from_pandas(val_df[["text"]])

In [16]:
# ── Trainer ───────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",   # Unsloth tokenizes internally
    packing=False,                # Unsloth handles packing
    max_seq_length=1024,         # back to 1024, Unsloth optimizes this
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # effective batch = 8
        num_train_epochs=1,
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        optim="adamw_8bit",
        fp16=True,
        dataloader_num_workers=0,
        report_to="wandb",
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=18,
        save_strategy="steps",
        save_steps=288,
        save_total_limit=10,
        output_dir="/kaggle/working/outputs",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
    ),
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2483 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/439 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,483 | Num Epochs = 1 | Total steps = 311
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 12,582,912 of 3,833,662,464 (0.33% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
18,No log,2.251458
36,2.383339,1.970741
54,2.109242,1.929702
72,1.930942,1.915570
90,1.928910,1.910740
108,1.925821,1.907381
126,1.912970,1.904383
144,1.954474,1.902807
162,1.903744,1.901252
180,1.912600,1.900271


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=311, training_loss=1.9662510421115102, metrics={'train_runtime': 5860.8832, 'train_samples_per_second': 0.424, 'train_steps_per_second': 0.053, 'total_flos': 3.940975997556941e+16, 'train_loss': 1.9662510421115102, 'epoch': 1.0})

In [17]:
# ── 8. SAVE & TEST ────────────────────────────────────────────
model.save_pretrained("phi-lora-model-posts")
tokenizer.save_pretrained("phi-lora-model-posts")

Unsloth: Restored added_tokens_decoder metadata in phi-lora-model-posts/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in phi-lora-model-posts.


('phi-lora-model-posts/tokenizer_config.json',
 'phi-lora-model-posts/chat_template.jinja',
 'phi-lora-model-posts/tokenizer.json')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"

ADAPTER_PATH = "/kaggle/working/phi-lora-model-posts"

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

# attach LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/working/phi-lora-model-posts"
)

In [24]:
from transformers import pipeline

from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=False,
    temperature=0.8,
    eos_token_id=tokenizer.eos_token_id,
)

prompt = """You are an expert Instagram content creator.

Topic: Watching the sunset on the beach

Write an Instagram caption that:

* Feels personal and authentic.
* Uses emojis naturally.
* Creates a visual image.
* Includes a question for followers.
* Includes 3-5 relevant hashtags.
* Is under 100 words.

Instagram Caption:

"""


result = pipe(prompt)
print(result[0]["generated_text"])

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You are an expert Instagram content creator.

Topic: Watching the sunset on the beach

Write an Instagram caption that:

* Feels personal and authentic.
* Uses emojis naturally.
* Creates a visual image.
* Includes a question for followers.
* Includes 3-5 relevant hashtags.
* Is under 100 words.

Instagram Caption:

"Just witnessed the most breathtaking sunset on the beach 🌅🌊 The colors were so vibrant, it felt like the sky was on fire! 🔥 I couldn't resist capturing this moment. 📸✨ It's moments like these that remind me to slow down and appreciate the beauty around us. 🌅💕 How do you find peace in nature


In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

prompt = """You are a professional content writer.

### Instruction:
Write a blog about "AI in healthcare" in a formal tone.

### Response:
"""

print(pipe(prompt)[0]["generated_text"])

In [27]:
import shutil

zip_file = shutil.make_archive(
    "/kaggle/working/phi-lora-model-posts",
    "zip",
    "/kaggle/working/phi-lora-model-posts"
)

In [28]:
print(zip_file)

/kaggle/working/phi-lora-model-posts.zip
